
# M2M-100 (zh->en) — Baseline TF Training (Without Wuxia Domain)

**TFG Hugo Silvosa – Baseline NMT (MarianMT)**  
Este cuaderno evalúa un modelo **MarianMT** (`facebook/m2m100_418M`) sin entrenar en dominio usando un dataset de **wuxia** (chino->inglés) ya preparado en formato `datasets` (HF).



## 1) Preparación del entorno

In [1]:

import os, random, math
import numpy as np

import torch
print("CUDA disponible:", torch.cuda.is_available())
print("Número de GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Nombre de la GPU:", torch.cuda.get_device_name(0))



CUDA disponible: True
Número de GPUs: 1
Nombre de la GPU: NVIDIA GeForce RTX 3060



> **Requisitos del dataset**: directorio HF Datasets con *splits* `train`, `validation`, `test` y columnas `zh` (chino) y `en` (inglés):  
> `processed_data/wuxia_zh_en_clean/`


In [2]:
# Configuración de carpetas para entorno LOCAL
from pathlib import Path
BASE_DIR = Path.cwd().parent.parent
BASE_DIR.mkdir(exist_ok=True)

for sub in ["evaluation", "models", "processed_data"]:
    (BASE_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Base:", BASE_DIR.resolve())
print("Estructura creada (si no existía):")
for p in ["evaluation", "models", "proccesed data"]:
    print(" -", (BASE_DIR / p).resolve())

# Nota: el dataset debe existir en: CORPUS/proccesed data/wuxia_zh_en_clean


Base: C:\Users\Usuario\Desktop\TFG\CORPUS
Estructura creada (si no existía):
 - C:\Users\Usuario\Desktop\TFG\CORPUS\evaluation
 - C:\Users\Usuario\Desktop\TFG\CORPUS\models
 - C:\Users\Usuario\Desktop\TFG\CORPUS\proccesed data


## 2) Configuración

In [3]:
from dataclasses import dataclass

@dataclass
class Config:
    # Rutas (local)
    dataset_dir: Path  = BASE_DIR / "processed_data" / "wuxia_zh_en_clean"   # <- carpeta con dataset HuggingFace guardado con load_from_disk
    output_dir: Path   = BASE_DIR / "models" / "pretrain_m2m100_wuxia"              # <- aquí se guardarán los runs/modelos
    ckpt_dir: Path     = BASE_DIR / "checkpoints"                          # <- checkpoints durante entrenamiento
    training_dir: Path = BASE_DIR / "training"         
    evaluation_dir: Path = BASE_DIR / "evaluation"
    translate_dir: Path = BASE_DIR / "evaluation" / "translate"
    translate_file: Path =   "pre_m2m100.txt"
    results_txt: Path = "pre_results.txt"
    
    # Columnas del dataset
    src_col: str = "zh"
    tgt_col: str = "en"

    # Idiomas M2M-100
    src_lang: str = "zh"
    tgt_lang: str = "en"

    # Modelo
    model_ckpt: str = "facebook/m2m100_418M"

    # Entrenamiento
    seed: int = 42
    max_source_length: int = 128
    max_target_length: int = 128
    batch_size: int = 16
    epochs: int = 10
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    early_stopping_patience: int = 3

    fraction: float = 1

cfg = Config()
print(cfg)


Config(dataset_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/processed_data/wuxia_zh_en_clean'), output_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/models/pretrain_m2m100_wuxia'), ckpt_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/checkpoints'), training_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/training'), evaluation_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/evaluation'), translate_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/evaluation/translate'), translate_file='pre_m2m100.txt', results_txt='pre_results.txt', src_col='zh', tgt_col='en', src_lang='zh', tgt_lang='en', model_ckpt='facebook/m2m100_418M', seed=42, max_source_length=128, max_target_length=128, batch_size=16, epochs=10, learning_rate=2e-05, weight_decay=0.01, early_stopping_patience=3, fraction=1)


In [4]:
import random, numpy as np, os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)



# Semillas para reproducibilidad
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)
os.environ["PYTHONHASHSEED"] = str(cfg.seed)


if device.type == "cuda":
    torch.cuda.manual_seed_all(cfg.seed)
    # Para reproducibilidad estricta 
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Semillas fijadas y backend configurado.")


Usando dispositivo: cuda
Semillas fijadas y backend configurado.


## 3) Cargar dataset (Hugging Face Datasets)

In [5]:

from datasets import load_from_disk, DatasetDict

assert os.path.isdir(cfg.dataset_dir), f"No se encuentra el dataset en: {cfg.dataset_dir}"
raw_ds: DatasetDict = load_from_disk(cfg.dataset_dir)
print(raw_ds)

# Validar columnas
def _check_cols(ds, src_col, tgt_col, split):
    cols = ds.column_names
    assert src_col in cols and tgt_col in cols, f"El split '{split}' debe contener columnas '{src_col}' y '{tgt_col}'. Columnas: {cols}"

for split in ["train", "validation", "test"]:
    assert split in raw_ds, f"Falta el split '{split}' en el dataset."
    _check_cols(raw_ds[split], cfg.src_col, cfg.tgt_col, split)

# pruebas rápidas
def take_fraction(ds, frac, seed=42):
    if frac >= 1.0:
        return ds
    n = max(1, int(len(ds) * frac))
    return ds.shuffle(seed=seed).select(range(n))

train_ds = take_fraction(raw_ds["train"], cfg.fraction, seed=cfg.seed)
val_ds   = take_fraction(raw_ds["validation"], cfg.fraction, seed=cfg.seed)
test_ds  = take_fraction(raw_ds["test"], cfg.fraction, seed=cfg.seed)

print(train_ds[:2])
print(f"Tam. train/val/test (fracción={cfg.fraction}):", len(train_ds), len(val_ds), len(test_ds))


DatasetDict({
    train: Dataset({
        features: ['zh', 'en'],
        num_rows: 417208
    })
    validation: Dataset({
        features: ['zh', 'en'],
        num_rows: 52151
    })
    test: Dataset({
        features: ['zh', 'en'],
        num_rows: 52151
    })
})
{'zh': ['第章 听到白小纯的话语，看到圣皇的迟疑，邪皇这里顿时呼吸一促，他目中刹那就露出凌厉之芒，右手猛的一挥，顿时那残破的红日，骤然幻化', '尤其是看到人群内的宋缺时，神算子立刻警惕，他当年在空城，是第一个跟随白小纯的，受到了重用，如今却成为了第二个，他顿时就视宋缺为竞争对手'], 'en': ['chapter- The Saint-Emperor hesitated, and the Vile-Emperor sucked in a breath, eyes flickering with cold light as he waved his hand to summon his damaged red sun', 'That was even more the case when he noticed Song Que among Bai Xiaochun’s men, which immediately got him even more on guard. Back in Sky City, Master God-Diviner had been the first to start following Bai Xiaochun again, and it had led to incredible benefits. Now, he was only the second to join him, which put Song Que in his sights as a major rival']}
Tam. train/val/test (fracción=1): 417208 52151 52151

## 5) Cargar tokenizador y modelo 

> **Nota:** Define `tokenizer.src_lang = 'zh'` y usa `forced_bos_token_id=tokenizer.get_lang_id('en')` en la generación para fijar el idioma destino.

In [6]:
from transformers import M2M100Tokenizer, M2M100ForConditionalGeneration
tokenizer = M2M100Tokenizer.from_pretrained(cfg.model_ckpt)
model = M2M100ForConditionalGeneration.from_pretrained(cfg.model_ckpt)
tokenizer.src_lang = cfg.src_lang
model.config.forced_bos_token_id = tokenizer.get_lang_id(cfg.tgt_lang)
model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Modelo cargado: {cfg.model_ckpt}")
print(f"Parámetros totales: {n_params:,}")

# Prueba 
sample = {
    'zh': '江湖夜雨十年灯。',
    'en': 'Ten years of lamps in the night rain of the jianghu.'
}
inputs = tokenizer(sample['zh'], return_tensors='pt').to(device)
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_length=cfg.max_target_length,
        num_beams=4, early_stopping=True,
        forced_bos_token_id=tokenizer.get_lang_id(cfg.tgt_lang)
    )
pred = tokenizer.decode(out[0], skip_special_tokens=True)
print('='*80)
print('ZH:', sample['zh'])
print('EN (ref):', sample['en'])
print('EN (pred):', pred)


Modelo cargado: facebook/m2m100_418M
Parámetros totales: 483,905,536


c:\Users\Usuario\miniconda3\envs\wuxia\Lib\site-packages\transformers\generation\utils.py:1715: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed in v5. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(


ZH: 江湖夜雨十年灯。
EN (ref): Ten years of lamps in the night rain of the jianghu.
EN (pred): A decade of rain light.


## 5) Preprocesamiento y tokenización

In [7]:
# Función para tokenizar ejemplos — M2M-100
def preprocess_function(examples):
    # fijar idiomas en cada llamada
    tokenizer.src_lang = cfg.src_lang          
    # Para tokenizar labels con text_target se usa tgt_lang
    try:
        tokenizer.tgt_lang = cfg.tgt_lang      
    except Exception:
        pass

    # Fuente
    model_inputs = tokenizer(
        examples[cfg.src_col],
        max_length=cfg.max_source_length,
        padding=False,
        truncation=True
    )

    try:
        labels = tokenizer(
            text_target=examples[cfg.tgt_col],
            max_length=cfg.max_target_length,
            padding=False,
            truncation=True
        )
    except TypeError:
        # Compatibilidad con transformers antiguos
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(
                examples[cfg.tgt_col],
                max_length=cfg.max_target_length,
                padding=False,
                truncation=True
            )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Aplicar tokenización a todo el dataset
tokenized_datasets = raw_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_ds["train"].column_names
)

train_ds = take_fraction(tokenized_datasets["train"], cfg.fraction, seed=cfg.seed)
val_ds   = take_fraction(tokenized_datasets["validation"], cfg.fraction, seed=cfg.seed)
test_ds  = take_fraction(tokenized_datasets["test"], cfg.fraction, seed=cfg.seed)

print(train_ds[0])


Map:   0%|          | 0/52151 [00:00<?, ? examples/s]

{'input_ids': [128102, 36741, 31659, 22, 18502, 1885, 10195, 2564, 89547, 36664, 29163, 4, 16501, 60116, 44253, 80, 117835, 21082, 4, 91101, 44253, 37256, 51147, 3720, 27321, 32282, 971, 51262, 4, 1957, 8257, 1203, 124909, 5133, 2354, 26528, 1799, 96983, 118223, 4326, 121938, 4, 49121, 2382, 104686, 9600, 116861, 4, 51147, 3720, 5133, 21035, 22105, 80, 26512, 1273, 4, 122102, 11318, 79929, 3453, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [128022, 74070, 436, 7, 1658, 12369, 7, 35701, 1215, 178, 30318, 1776, 241, 4, 1019, 1197, 141, 421, 7, 35701, 1215, 178, 248, 119237, 28, 8, 120052, 4, 122301, 109150, 918, 150, 9792, 124249, 86770, 285, 1307, 84848, 241, 14810, 3972, 128, 4088, 5722, 14810, 9627, 117626, 5091, 4292, 2]}


## 6) Evaluación 

In [8]:
import os
import json
import time
from tqdm.auto import tqdm
import sacrebleu
from sacrebleu.metrics import CHRF, TER
from rouge_score import rouge_scorer
import nltk
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import wordpunct_tokenize
import numpy as np
import torch
import logging

from bert_score import score as calc_bertscore
from comet import download_model, load_from_checkpoint

start = time.time()
# Descargar recursos de NLTK para METEOR
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

# Ocultar advertencias molestas de PyTorch Lightning (usado por COMET)
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)

# Parámetros 
EVAL_MAX_SAMPLES = 1000        # None = todo el split
PRED_BEAMS = 4
BATCH_EVAL = max(1, cfg.batch_size // 2)

# Comprobaciones 
assert 'model' in globals(), "No se encontró `model`. Carga el modelo antes."
assert 'tokenizer' in globals(), "No se encontró `tokenizer`. Cárgalo antes."
assert 'val_ds' in globals() and 'test_ds' in globals(), "Faltan `val_ds` y/o `test_ds`."
assert 'cfg' in globals(), "Falta `cfg`."

# Asegurar pad_token_id
if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

eval_raw = test_ds if len(test_ds) > 0 else val_ds
n_total = len(eval_raw)
n_eval = n_total if (EVAL_MAX_SAMPLES is None) else min(n_total, int(EVAL_MAX_SAMPLES))
assert n_eval > 0, "No hay ejemplos para evaluar."

def decode_ids_to_text(dataset, id_col):
    return [
        tokenizer.decode(ids, skip_special_tokens=True)
        for ids in dataset[id_col]
    ]

src_texts = decode_ids_to_text(eval_raw, "input_ids")[:n_eval]
ref_texts = decode_ids_to_text(eval_raw, "labels")[:n_eval]

# Generación por lotes
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def batched_generate(texts, batch_size=8, max_length=128, num_beams=4):
    preds = []
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Generando traducciones"):
            batch = texts[i:i+batch_size]
            inputs = tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=cfg.max_source_length
            ).to(device)
            outputs = model.generate(
                **inputs,
                max_length=max_length,
                num_beams=num_beams,
                early_stopping=True,
                forced_bos_token_id=tokenizer.get_lang_id(cfg.tgt_lang)
            )
            preds.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))
    return preds

preds = batched_generate(
    src_texts,
    batch_size=BATCH_EVAL,
    max_length=cfg.max_target_length,
    num_beams=PRED_BEAMS
)

model.cpu()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# #  Métricas 
# print("Calculando métricas clásicas...")
# bleu_corpus = sacrebleu.corpus_bleu(preds, [ref_texts]).score

# chrf_metric = CHRF(word_order=2)
# chrf_corpus = chrf_metric.corpus_score(preds, [ref_texts]).score

# ter_metric = TER()
# ter_corpus = ter_metric.corpus_score(preds, [ref_texts]).score

# def compute_rougeL_f1(hyp_list, ref_list):
#     scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
#     f1s = []
#     for h, r in zip(hyp_list, ref_list):
#         s = scorer.score(r, h)
#         f1s.append(s['rougeL'].fmeasure)
#     return float(np.mean(f1s)) * 100.0
# rougeL_f1 = compute_rougeL_f1(preds, ref_texts)

# def compute_meteor(hyp_list, ref_list):
#     scores = []
#     for hyp, ref in zip(hyp_list, ref_list):
#         hyp_tok = wordpunct_tokenize(hyp) if isinstance(hyp, str) else hyp
#         ref_tok = wordpunct_tokenize(ref) if isinstance(ref, str) else ref
#         scores.append(meteor_score([ref_tok], hyp_tok))
#     return float(np.mean(scores)) * 100.0
# meteor_avg = compute_meteor(preds, ref_texts)

# --- BERTSCORE ---
print("Calculando BERTScore...")
# Extraemos el idioma destino desde cfg (si no lo tienes configurado así, cámbialo por "en", "es", etc.)
tgt_language = getattr(cfg, "tgt_lang", "multilingual")
_, _, F1 = calc_bertscore(preds, ref_texts, lang=tgt_language, verbose=True, rescale_with_baseline=True)
bertscore_avg = float(F1.mean()) * 100.0

# --- COMET ---
print("Calculando COMET (puede tardar un poco en cargar la primera vez)...")
comet_model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(comet_model_path)
comet_model.eval()

# COMET necesita el texto original (src), la predicción (mt) y la referencia (ref)
comet_data = [
    {"src": s, "mt": p, "ref": r} 
    for s, p, r in zip(src_texts, preds, ref_texts)
]
comet_output = comet_model.predict(comet_data, batch_size=BATCH_EVAL, progress_bar=True)
comet_avg = float(comet_output.system_score) * 100.0

end_time = time.time()

results = {
    "model" : getattr(cfg, "model_ckpt", "unknown_model"),
    "n_eval": n_eval,
    "num_beams": PRED_BEAMS,
    "batch_eval": BATCH_EVAL,
    # "sacrebleu": round(bleu_corpus, 4),
    # "chrf2": round(chrf_corpus, 4),
    # "ter": round(ter_corpus, 4),
    # "rougeL_f1": round(rougeL_f1, 4),
    # "meteor": round(meteor_avg, 4), 
    "bertscore": round(bertscore_avg, 4),
    "comet": round(comet_avg, 4),
    "execution_time": round(end_time - start, 2)
}

if hasattr(cfg, "evaluation_dir"):
    os.makedirs(cfg.evaluation_dir, exist_ok=True)
    res_file = os.path.join(cfg.evaluation_dir, getattr(cfg, "results_file", "results.json"))
    with open(res_file, "a", encoding="utf-8") as f:
        f.write("\n")
        f.write(json.dumps(results, ensure_ascii=False, indent=4))

print("\n--- RESULTADOS FINALES ---")
print(json.dumps(results, indent=4))

c:\Users\Usuario\miniconda3\envs\wuxia\Lib\site-packages\torchmetrics\utilities\imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Generando traducciones:   0%|          | 0/125 [00:00<?, ?it/s]

Calculando BERTScore...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


...

  0%|          | 0/16 [00:00<?, ?it/s]

done in 5.85 seconds, 170.81 sentences/sec
Calculando COMET (puede tardar un poco en cargar la primera vez)...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Encoder model frozen.
c:\Users\Usuario\miniconda3\envs\wuxia\Lib\site-packages\pytorch_lightning\core\saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
Predicting DataLoader 0: 100%|██████████| 125/125 [00:12<00:00,  9.80it/s]



--- RESULTADOS FINALES ---
{
    "model": "facebook/m2m100_418M",
    "n_eval": 1000,
    "num_beams": 4,
    "batch_eval": 8,
    "bertscore": 25.9788,
    "comet": 56.1681,
    "execution_time": 442.53
}


In [9]:
from bert_score import score

# Your sentences
cands = ["It's not enough. You're a wise man who better understands the situation."]
refs = ["Friend Suan Bu Jin, as a wisdom path Gu Immortal, you should be even more aware of the current situation"]

# Calculate BERTScore (using the default English model, roberta-large)
P, R, F1 = score(cands, refs, lang="en", verbose=True, rescale_with_baseline=True)



print(f"Precision: {P.item():.4f}")
print(f"Recall: {R.item():.4f}")
print(f"F1 Score: {F1.item():.4f}")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.09 seconds, 11.62 sentences/sec
Precision: 0.3418
Recall: 0.1226
F1 Score: 0.2311


## 7) Muestra cualitativa (n ejemplos aleatorios)

In [15]:
import random
import torch

# Mostrar predicciones aleatorias 
n_show = 100
idxs = random.sample(range(len(eval_raw)), k=min(n_show, len(eval_raw)))

model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for i in idxs:
    # Decodificar texto fuente y referencia desde dataset tokenizado
    zh = tokenizer.decode(eval_raw[i]["input_ids"], skip_special_tokens=True)
    en_ref = tokenizer.decode(eval_raw[i]["labels"], skip_special_tokens=True)

    # Tokenizar entrada y mover a dispositivo
    inputs = tokenizer(zh, return_tensors="pt").to(device)

    # Generar traducción
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_length=cfg.max_target_length,
            num_beams=4,
            early_stopping=True,
            forced_bos_token_id=tokenizer.get_lang_id(cfg.tgt_lang)
        )

    en_pred = tokenizer.decode(out[0], skip_special_tokens=True)

    print("="*80)
    print("ZH:", zh)
    print("EN (ref):", en_ref)
    print("EN (pred):", en_pred)


ZH: 最关键的是——真正的天庭,不是在这里,而是在我们的心中
EN (ref): And most importantly, the true Heavenly Court is not here, it is within our hearts
EN (pred): The most important thing is that the true heaven is not here, but in our hearts.
ZH: 这一切他都清楚,可他还是脚步停顿下来,没有选择离去,因为 他不能错过这个机会,一旦错过,哪怕是在这炼魂壶的世界内,他想要去找到白浩,也都极为艰难,且一旦在这过程中,白浩的魂出现了灭亡,那么将是他一生的遗憾
EN (ref): He could not pass up this oppor tunity. After all, even knowing that Bai Hao’s soul was in the Necromancer Kettle’t necessarily do much good. Even going to search for him in that one particular area would be difficult. Furthermore, if Bai Hao’s soul were to die, then Bai Xiaochun would feel regret over the matter for the rest of his life
EN (pred): All this he knew, but he still stopped, no choice to leave, because he could not miss this opportunity, once missed, even in the world of this refinery, he wanted to find White, and it was very difficult, and once in this process, White's soul appeared to disappear, then it would be his whole life regret.
ZH: 我不信


In [16]:
from tqdm import tqdm

os.makedirs(cfg.translate_dir, exist_ok=True)
translate_path = os.path.join(cfg.translate_dir, cfg.translate_file)

with open(translate_path, "w", encoding="utf-8") as f:
    for i in tqdm(range(len(eval_raw) // 2)):
        zh = tokenizer.decode(eval_raw[i]["input_ids"], skip_special_tokens=True)
        en_ref = tokenizer.decode(eval_raw[i]["labels"], skip_special_tokens=True)

        inputs = tokenizer(zh, return_tensors="pt").to(device)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_length=cfg.max_target_length,
                num_beams=4,
                early_stopping=True,
                forced_bos_token_id=tokenizer.get_lang_id(cfg.tgt_lang)
            )

        en_pred = tokenizer.decode(out[0], skip_special_tokens=True)

        # Guardar en el archivo
        f.write("="*80 + "\n")
        f.write("ZH: " + zh + "\n")
        f.write("EN (ref): " + en_ref + "\n")
        f.write("EN (pred): " + en_pred + "\n\n")


100%|██████████| 26075/26075 [3:31:14<00:00,  2.06it/s]  
